In [2]:
import torch
import torch.nn as nn
import numpy as np

# 1. Synthetic data generator

def gen_synth_batch(batch_sz=32, seq_len=24, feat_dim=2):
    # generates dummy hourly fitbit data and missingness masks
    # base physiological signal (e.g. sin waves for circadian rhythms)
    t = torch.linspace(0, 4 * np.pi, seq_len).unsqueeze(0).unsqueeze(-1)
    base_signal = torch.sin(t).expand(batch_sz, seq_len, feat_dim)

    # add noise
    x_fitbit = base_signal + torch.randn(batch_sz, seq_len, feat_dim) * 0.2

    # generate missingness mask (delta). 1 = observed, 0 = missing
    # simulate missingness: periods of high signal variance more likely to be missing
    mask_prob = torch.sigmoid(torch.var(x_fitbit, dim=-1, keepdim=True) - 0.5)
    delta_mask = torch.bernoulli(mask_prob)

    # apply mask to data (zero out missing values)
    x_fitbit = x_fitbit * delta_mask

    # dummy EHR vector
    x_ehr = torch.randn(batch_sz, 5)

    return x_fitbit, delta_mask, x_ehr

# 2. T-CRL architecture components

class TCN_Block(nn.Module):
    # 1D convolution for temporal processing
    def __init__(self, in_chan, out_chan, kernel_sz=3):
        super().__init__()
        self.conv = nn.Conv1d(in_chan, out_chan, kernel_sz, padding=kernel_sz//2)
        self.relu = nn.ReLU()

    def forward(self, x):
        # pytorch conv1d expects (batch, channels, seq_len)
        x = x.transpose(1, 2)
        out = self.relu(self.conv(x))
        return out.transpose(1, 2) # back to (batch, seq, channels)

class MissingnessFusionGate(nn.Module):
    # sigmoid mechanism to weigh inputs based on gaps
    def __init__(self, mask_dim, hidden_dim):
        super().__init__()
        self.mask_mlp = nn.Sequential(
            nn.Linear(mask_dim, hidden_dim),
            nn.Sigmoid()
        )

    def forward(self, h_temporal, delta_mask):
        # learn embedding from binary mask
        gate_weights = self.mask_mlp(delta_mask)
        # gate temporal features
        return h_temporal * gate_weights

class TCRL_Encoder(nn.Module):
    # full stage 1 multimodal encoder
    def __init__(self, feat_dim=2, seq_len=24, ehr_dim=5, hidden_dim=16):
        super().__init__()
        # fitbit temporal encoder
        self.tcn = TCN_Block(feat_dim, hidden_dim)

        # missingness gate
        self.fusion_gate = MissingnessFusionGate(mask_dim=1, hidden_dim=hidden_dim)

        # EHR embedding (simplified mean-pooling)
        self.ehr_mlp = nn.Sequential(
            nn.Linear(ehr_dim, hidden_dim),
            nn.ReLU()
        )

        # projection to latent space Z
        self.to_latent = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, x_fitbit, delta_mask, x_ehr):
        # process temporal data
        h_temp = self.tcn(x_fitbit)

        # apply missingness gate
        h_gated = self.fusion_gate(h_temp, delta_mask)

        # pool temporal dimension (mean over 24 hrs)
        h_gated_pooled = torch.mean(h_gated, dim=1)

        # process static EHR events
        h_ehr = self.ehr_mlp(x_ehr)

        # fuse modalities
        fused_rep = torch.cat([h_gated_pooled, h_ehr], dim=-1)
        z_latent = self.to_latent(fused_rep)

        return z_latent

# 3. Execution/test

if __name__ == "__main__":
    # simulate batch of 32 patients, 24 hours of data, 2 features (HR, Steps)
    x_fit, mask, x_ehr = gen_synth_batch(batch_sz=32, seq_len=24, feat_dim=2)

    model = TCRL_Encoder()
    z_out = model(x_fit, mask, x_ehr)

    print("Model initialized")
    print(f"Input shape (Fitbit): {x_fit.shape}")
    print(f"Output latent space shape: {z_out.shape}")

Model initialized
Input shape (Fitbit): torch.Size([32, 24, 2])
Output latent space shape: torch.Size([32, 16])
